# Track 2: Atlas-Free Coordinate GNN — A100 Training

**Hardware**: A100 40GB — Runtime → Change runtime type → A100 GPU  
**Output**: `best_coord_gnn.pt` + `last_coord_gnn.pt` → Google Drive

### Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Run all cells top-to-bottom.

### Resuming after a session crash
Re-run all cells from the top. A single packed graph cache is copied from Drive to local SSD automatically, so Colab does not scan thousands of tiny graph files.


## 0 — Runtime check

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError('No GPU — Runtime → Change runtime type → A100 GPU')
print(gpu_info.stdout)

import torch
assert torch.cuda.is_available()
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1 — Install dependencies

In [ ]:
import torch, sys, subprocess
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')
torch_tag = torch.__version__.split('+')[0]
print(f'torch={torch_tag}  cuda={cuda_tag}')

In [ ]:
# Minimal installs — coordinates are a parquet, text embeddings are pre-computed .pt files.
# No nimare / transformers / adapters needed.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow', 'pandas', 'matplotlib', 'tqdm', 'umap-learn'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel', 'nilearn'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric'], check=True)
pyg_url = f'https://data.pyg.org/whl/torch-{torch_tag}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_scatter', 'torch_sparse', 'torch_cluster', '-f', pyg_url], check=True)
print('Done.')

## 2 — Clone repo + patch __init__

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/neurovlm')
if not REPO_DIR.exists():
    !git clone --branch neurovlm_gnn https://github.com/neurovlm/neurovlm.git {REPO_DIR}
else:
    print('Repo exists — pulling.')
    !git -C {REPO_DIR} pull

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

(REPO_DIR / 'src/neurovlm/__init__.py').write_text(
    'from __future__ import annotations\n\n\n'
    'def __getattr__(name: str):\n'
    '    if name == "NeuroVLM":\n'
    '        from .core import NeuroVLM\n'
    '        return NeuroVLM\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n'
)
print('sys.path:', src_dir)

## 3 — Mount Google Drive + set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR      = Path('/content/drive/MyDrive/neurovlm_track2')
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints/coord_gnn'
DRIVE_DATA_DIR = DRIVE_DIR / 'data'                 # persistent across sessions
LOCAL_DATA_DIR = Path('/content/neurovlm_track2')   # fast local SSD, lost on crash

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive data   → {DRIVE_DATA_DIR}')
print(f'Local SSD    → {LOCAL_DATA_DIR}')
print(f'Checkpoints  → {CHECKPOINT_DIR}')


## 4 — Hyperparameters (A100-tuned)

In [ ]:
CFG = dict(
    K            = 7,
    MAX_DIST_MM  = 30.0,
    HIDDEN       = 256,    # ~10M params — larger than local default of 128
    HEADS        = 8,
    DROPOUT      = 0.1,
    N_EPOCHS     = 200,
    BATCH_SIZE   = 512,    # 512 graphs × ~20 avg nodes = ~10K nodes/batch
    LR_GNN       = 1e-4,
    LR_PROJ      = 1e-5,
    WARMUP_EPOCHS= 15,
    TEMPERATURE  = 0.07,
    VAL_INTERVAL = 5,
    NUM_WORKERS  = 4,
    PREFETCH_FACTOR = 4,
    PIN_MEMORY   = True,
    USE_AMP      = True,   # BF16 on A100 Tensor Cores
    SEED         = 42,
)
print('Config:', CFG)


## 5 — Load coordinates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from neurovlm.retrieval_resources import _load_pubmed_coordinates
from neurovlm.gnn.coord_graph import normalize_coords

print('Loading peak coordinates …')
coords_df = _load_pubmed_coordinates()
print(f'  {coords_df.shape}  columns: {list(coords_df.columns)}')

peak_counts = coords_df.groupby('pmid').size()
print(f'\nPeak counts across {len(peak_counts):,} papers:')
print(f'  min={peak_counts.min()}  max={peak_counts.max()}')
print(f'  mean={peak_counts.mean():.1f}  median={peak_counts.median():.0f}')
print(f'  p5={peak_counts.quantile(0.05):.0f}  p95={peak_counts.quantile(0.95):.0f}')

## 6 — Load text embeddings

In [ ]:
from neurovlm.data import load_dataset, load_latent

print('Loading SPECTER text embeddings …')
text_latents = load_latent('pubmed_text')
pubmed_df    = load_dataset('pubmed_text')

if isinstance(text_latents, tuple):
    text_tensor, pmids_arr = text_latents
    unique_pmids = np.asarray(pmids_arr).astype(str)
elif isinstance(text_latents, dict):
    pmid_list    = list(text_latents.keys())
    text_tensor  = torch.stack([torch.tensor(text_latents[p], dtype=torch.float32)
                                for p in pmid_list])
    unique_pmids = np.array([str(p) for p in pmid_list])
else:
    text_tensor  = text_latents if isinstance(text_latents, torch.Tensor) \
                   else torch.tensor(text_latents)
    unique_pmids = pubmed_df['pmid'].astype(str).values[:len(text_tensor)] \
                   if 'pmid' in pubmed_df.columns \
                   else np.arange(len(text_tensor)).astype(str)

print(f'  text_tensor  : {text_tensor.shape}')
print(f'  unique_pmids : {len(unique_pmids):,}')

## 7 — Load or build the packed graph cache

**Key design**: Colab/Drive is slow at thousands of tiny `paper_*.pt` files. This notebook now uses one packed cache file per graph configuration.

- Drive is touched only for one file copy in and one file copy out.
- Training reads graphs from RAM, not Drive or local disk.
- The old per-paper cache directory is intentionally bypassed.


In [ ]:
import shutil

max_dist_tag = str(CFG['MAX_DIST_MM']).replace('.', 'p')
cache_tag = f'coord_graphs_k{CFG["K"]}_d{max_dist_tag}.pt'
DRIVE_CACHE_FILE = DRIVE_DATA_DIR / cache_tag
LOCAL_CACHE_FILE = LOCAL_DATA_DIR / cache_tag

if DRIVE_CACHE_FILE.exists() and not LOCAL_CACHE_FILE.exists():
    print('Copying packed graph cache from Drive → local SSD ...')
    shutil.copy2(DRIVE_CACHE_FILE, LOCAL_CACHE_FILE)
elif LOCAL_CACHE_FILE.exists():
    print('Packed graph cache already exists on local SSD.')
else:
    print('No packed graph cache found; the next cell will build it once on local SSD.')

print(f'Drive cache file → {DRIVE_CACHE_FILE}')
print(f'Local cache file → {LOCAL_CACHE_FILE}')


In [ ]:
# Build/load one packed cache file on local SSD. When present, this is one torch.load.
from neurovlm.gnn.coord_dataset import CoordGraphDataset

print(f'Loading packed graph cache (k={CFG["K"]}, max_dist={CFG["MAX_DIST_MM"]}mm) ...')
ds = CoordGraphDataset(
    coords_df       = coords_df,
    text_embeddings = text_tensor,
    unique_pmids    = unique_pmids,
    cache_dir       = str(LOCAL_DATA_DIR / 'legacy_coord_graphs'),
    cache_file      = str(LOCAL_CACHE_FILE),
    k               = CFG['K'],
    max_dist_mm     = CFG['MAX_DIST_MM'],
    preload_to_ram  = True,
)
print(f'Dataset size: {len(ds):,} papers')


In [ ]:
# Persist the packed cache back to Drive for the next Colab session.
# This copies exactly one file, avoiding Drive directory scans.
if LOCAL_CACHE_FILE.exists():
    if (not DRIVE_CACHE_FILE.exists()
        or LOCAL_CACHE_FILE.stat().st_size != DRIVE_CACHE_FILE.stat().st_size):
        print('Copying packed graph cache local SSD → Drive ...')
        shutil.copy2(LOCAL_CACHE_FILE, DRIVE_CACHE_FILE)
    else:
        print('Drive packed cache is already up to date.')
print('Packed cache ready.')


In [ ]:
# Verify PyG batching. Graphs are already in RAM when using cache_file.
from torch_geometric.loader import DataLoader
probe = next(iter(DataLoader(ds, batch_size=4, shuffle=False)))
print(f'\nBatch check (size=4):')
print(f'  x.shape         = {probe.x.shape}   (total_nodes, 5)')
print(f'  edge_attr.shape = {probe.edge_attr.shape}  (E, 4)')
print(f'  y.shape         = {probe.y.shape}')
assert probe.x.shape[1] == 5 and probe.edge_attr.shape[1] == 4
print('✓ shapes correct')


In [ ]:
torch.manual_seed(CFG['SEED'])
train_ds, val_ds, test_ds = ds.split(val_frac=0.1, test_frac=0.1, seed=CFG['SEED'])
print(f'Split: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

## 8 — Model

In [ ]:
from torch_geometric.data import Data, Batch
from neurovlm.gnn.coord_model import CoordGNN
from neurovlm.gnn.model import TextProjHead

brain_encoder = CoordGNN(
    in_dim=5, hidden=CFG['HIDDEN'], heads=CFG['HEADS'],
    out_dim=384, dropout=CFG['DROPOUT'],
)
text_proj = TextProjHead(in_dim=768, hidden_dim=512, out_dim=384)

n_brain = brain_encoder.count_parameters()
n_text  = sum(p.numel() for p in text_proj.parameters())
print(f'CoordGNN params     : {n_brain:,}')
print(f'TextProjHead params : {n_text:,}')

dummy = Data(x=torch.randn(10,5),
             edge_index=torch.tensor([[0,1,2],[1,2,0]], dtype=torch.long),
             edge_attr=torch.randn(3,4))
b = Batch.from_data_list([dummy, dummy])
out = brain_encoder(b.x, b.edge_index, b.edge_attr, b.batch)
assert out.shape == (2, 384)
print(f'✓ Forward pass: (20,5) → {tuple(out.shape)}')

In [ ]:
# torch.compile with 'default' mode — fuses ops without CUDAGraphs.
# 'reduce-overhead' (CUDAGraphs) breaks on variable-size GNN batches.
try:
    brain_encoder = torch.compile(brain_encoder, mode='default')
    text_proj     = torch.compile(text_proj,     mode='default')
    print('torch.compile enabled (mode=default).')
except Exception as e:
    print(f'torch.compile skipped: {e}')

## 9 — Training

**A100 optimizations active:**
- packed graph cache — one `torch.load`, no per-paper cache scan
- RAM graph dataset — zero file I/O per batch
- `batch_size=512` — 512 graphs/batch fills GPU compute
- `num_workers=4`, `prefetch_factor=4`, `pin_memory=True` — parallel CPU prefetch + fast CPU→GPU transfer
- `use_amp=True` — BF16 on A100 Tensor Cores (~2x vs FP32)


In [ ]:
torch.cuda.reset_peak_memory_stats()
print(f'VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
from neurovlm.gnn.coord_train import CoordTrainer

trainer = CoordTrainer(
    brain_encoder  = brain_encoder,
    text_proj      = text_proj,
    lr_gnn         = CFG['LR_GNN'],
    lr_proj        = CFG['LR_PROJ'],
    batch_size     = CFG['BATCH_SIZE'],
    n_epochs       = CFG['N_EPOCHS'],
    warmup_epochs  = CFG['WARMUP_EPOCHS'],
    temperature    = CFG['TEMPERATURE'],
    device         = 'cuda',
    val_interval   = CFG['VAL_INTERVAL'],
    checkpoint_dir = str(CHECKPOINT_DIR),
    num_workers    = CFG['NUM_WORKERS'],
    prefetch_factor= CFG['PREFETCH_FACTOR'],
    pin_memory     = CFG['PIN_MEMORY'],
    use_amp        = CFG['USE_AMP'],
    verbose        = True,
)

trainer.fit(train_ds, val_ds)


In [ ]:
peak_gb  = torch.cuda.max_memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Peak VRAM: {peak_gb:.2f} / {total_gb:.1f} GB  ({100*peak_gb/total_gb:.1f}%)')
if peak_gb / total_gb < 0.5:
    print('VRAM <50% — try BATCH_SIZE=1024 or HIDDEN=384 next run.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(trainer.history['train_loss'])
axes[0].set_title('Train Loss (InfoNCE)'); axes[0].set_xlabel('Epoch')
axes[1].plot(trainer.history['val_recall_t2i'], label='t2i')
axes[1].plot(trainer.history['val_recall_i2t'], label='i2t')
axes[1].set_title('Val Recall AUC'); axes[1].legend()
axes[2].plot(trainer.history['embed_sim'], color='red')
axes[2].axhline(0.95, color='black', ls='--', label='collapse threshold')
axes[2].set_title('embed_sim (collapse monitor)'); axes[2].legend()
plt.tight_layout()
plt.savefig(str(DRIVE_DIR / 'training_curve.png'), dpi=150)
plt.show()

## 10 — Evaluation

In [ ]:
from neurovlm.metrics import recall_at_k, recall_curve

trainer.restore_best()
results = {}

for split_name, split_ds in [('Val', val_ds), ('Test', test_ds)]:
    brain_emb, text_emb = trainer.collect_embeddings(split_ds)
    sim = F.normalize(text_emb, dim=1) @ F.normalize(brain_emb, dim=1).T
    r1  = recall_at_k(sim, 1)
    r5  = recall_at_k(sim, 5)
    r10 = recall_at_k(sim, 10)
    t2i, i2t = recall_curve(text_emb, brain_emb)
    auc = float((t2i.mean() + i2t.mean()) / 2)
    results[split_name] = dict(r1=r1, r5=r5, r10=r10, auc=auc)
    print(f'\n{split_name} ({len(split_ds)} papers):')
    print(f'  recall@1={r1:.4f}  recall@5={r5:.4f}  recall@10={r10:.4f}  AUC={auc:.4f}')

print('\n  NeuroVLM MLP baseline AUC ≈ 0.81')

## 11 — Attention analysis

In [ ]:
snapshots = trainer.get_attention_snapshot(test_ds, n_samples=10)
for snap in snapshots:
    n_peaks = snap['node_coords_mni'].shape[0]
    print(f'\nPaper {snap["paper_idx"]} ({n_peaks} peaks):')
    for e in snap['top_edges']:
        dist_mm = float(np.linalg.norm(np.array(e['src_mni']) - np.array(e['dst_mni'])))
        s = [f'{v:.0f}' for v in e['src_mni']]
        d = [f'{v:.0f}' for v in e['dst_mni']]
        print(f'  [{" ".join(s)}] → [{" ".join(d)}]  attn={e["weight"]:.4f}  dist={dist_mm:.0f}mm')

## 12 — UMAP + save

In [ ]:
try:
    import umap
    brain_emb, _ = trainer.collect_embeddings(test_ds)
    emb_np = F.normalize(brain_emb, dim=1).cpu().numpy()
    N = min(500, len(emb_np))
    idx = np.random.default_rng(42).choice(len(emb_np), N, replace=False)
    emb_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(emb_np[idx])
    fig, ax = plt.subplots(figsize=(8,6))
    ax.scatter(emb_2d[:,0], emb_2d[:,1], s=5, alpha=0.6, c='steelblue')
    ax.set_title(f'UMAP — CoordGNN ({N} test papers)')
    plt.tight_layout()
    plt.savefig(str(DRIVE_DIR / 'umap_coord_gnn.png'), dpi=150)
    plt.show()
except ImportError:
    print('umap-learn not available.')

In [ ]:
import json
with open(DRIVE_DIR / 'training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals]
               for k, vals in trainer.history.items()}, f, indent=2)
with open(DRIVE_DIR / 'eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved to Drive:')
for p in sorted(DRIVE_DIR.rglob('*')):
    if p.is_file() and not str(p).endswith('.pt'):
        print(f'  {p.relative_to(DRIVE_DIR)}')
# Print checkpoint sizes separately
for p in CHECKPOINT_DIR.glob('*.pt'):
    print(f'  checkpoints/{p.name}  ({p.stat().st_size/1e6:.1f} MB)')